# Dashboard Data Preparation

After generating predictions using the machine learning pipeline, the next step is to prepare the data for visualization in the interactive dashboard.

Raw prediction outputs are not directly suitable for user interfaces. Therefore, additional transformations are required to:

- Aggregate predictions at the city level
- Generate summary metrics for visualization
- Create formatted labels for map popups
- Structure data for interactive components such as dropdowns and plots

This notebook prepares the dataset in a format that aligns directly with the Shiny dashboard backend logic.

The output of this notebook mirrors the structure used in the `server.R` file, ensuring seamless integration.

## Import Required Libraries

We import libraries required for:

- Data manipulation and aggregation
- String formatting for labels
- Handling grouped operations

In [ ]:
# Import pandas for structured data manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

## Load Prediction Data

We load the dataset generated from the prediction pipeline.

This dataset contains:

- City information (if included)
- Weather variables
- Predicted bike demand
- Demand levels
- Datetime values

In [ ]:
# Load prediction dataset generated in Notebook 10
df = pd.read_csv("predicted_bike_demand.csv")

# Display first few rows to inspect structure
df.head()

## Aggregate Data at City Level

The dashboard map displays one marker per city, representing the maximum predicted bike demand over the forecast period.

We group the data by city and compute:

- Maximum predicted demand
- Corresponding demand level
- Associated metadata for display

In [ ]:
# Group data by city and location coordinates
cities_max_bike = df.loc[
    df.groupby("CITY_ASCII")["bike_prediction"].idxmax()
].reset_index(drop=True)

## Create Dashboard Labels

The dashboard requires two types of labels:

### 1. LABEL (Overview Map)
- Short summary for each city

### 2. DETAILED_LABEL (Drill-down View)
- Includes temperature, humidity, datetime, and prediction details

These labels are formatted using HTML for display in Leaflet popups.

In [ ]:
# Create short label for overview map
cities_max_bike["LABEL"] = (
    "<b>" + cities_max_bike["CITY_ASCII"] + "</b><br/>" +
    "Demand: " + cities_max_bike["bike_prediction"].round(0).astype(str)
)

# Create detailed label for selected city view
cities_max_bike["DETAILED_LABEL"] = (
    "<b style='font-size:16px;'>" + cities_max_bike["CITY_ASCII"] + "</b><br/>" +
    "<hr/>" +
    "<b>Bike Demand:</b> " + cities_max_bike["bike_prediction"].round(0).astype(str) + "<br/>" +
    "<b>Temperature:</b> " + cities_max_bike["temperature"].round(1).astype(str) + " °C<br/>" +
    "<b>Humidity:</b> " + cities_max_bike["humidity"].round(1).astype(str) + " %<br/>" +
    "<b>Forecast Time:</b><br/>" + cities_max_bike["datetime"].astype(str)
)

## Map Demand Levels to Visualization Attributes

To enhance visualization:

- Marker size and color depend on demand level
- These categories will be used in the Leaflet map

Mapping:

- small → green
- medium → yellow
- large → red

In [ ]:
# Define mapping for visualization attributes
def map_color(level):
    if level == "small":
        return "green"
    elif level == "medium":
        return "yellow"
    else:
        return "red"

def map_radius(level):
    if level == "small":
        return 6
    elif level == "medium":
        return 10
    else:
        return 12

# Apply mappings
cities_max_bike["color"] = cities_max_bike["demand_level"].apply(map_color)
cities_max_bike["radius"] = cities_max_bike["demand_level"].apply(map_radius)

## Final Dataset for Dashboard

The final dataset includes:

- City name and coordinates
- Predicted demand
- Demand level
- Visualization attributes
- Popup labels

This dataset is now ready for integration into the Shiny dashboard.

In [ ]:
# Save prepared dataset for dashboard use
cities_max_bike.to_csv("dashboard_ready_data.csv", index=False)

## Summary

In this notebook, we:

- Aggregated prediction data at the city level
- Generated formatted labels for visualization
- Created mapping logic for marker styling
- Prepared a structured dataset for dashboard integration

This step ensures that the prediction outputs are fully compatible with the interactive dashboard interface.

## Integration Note: Relationship Between Notebook and Shiny Dashboard

This notebook demonstrates the data preparation logic required to support the interactive dashboard. However, it is important to clarify how this notebook relates to the deployed Shiny application.

The Shiny dashboard is implemented entirely in **R**, and the backend logic for:

- Data aggregation  
- Feature transformation  
- Prediction generation  
- Label creation  

is handled within the following core files:

- `server.R`
- `model_prediction.R`

These components represent the **primary implementation of the project**, developed as part of the capstone lab using the R programming language.

---

### Purpose of This Notebook

This notebook serves as an **enhancement layer** to the core project by:

- Reconstructing backend logic in a step-by-step analytical format  
- Providing transparency into how prediction outputs are structured  
- Improving accessibility for learners familiar with Python-based workflows  
- Demonstrating cross-language implementation of the same pipeline  

---

### Important Distinction

- The Shiny dashboard operates entirely on **R-based code**
- This notebook is **not executed within the Shiny application**
- Instead, it mirrors the same logic using Python for clarity and extended analysis

---

### Why This Dual-Layer Approach Is Valuable

Maintaining both R and Python implementations provides several advantages:

- Reinforces understanding by expressing the same pipeline in two ecosystems  
- Enhances the project’s appeal to a broader technical audience  
- Demonstrates flexibility in both statistical (R) and general-purpose (Python) data science environments  
- Strengthens the portfolio by showcasing multi-language proficiency  

---

### Project Positioning

- **R Implementation** → Primary system (Capstone + Shiny Dashboard)  
- **Python Notebooks** → Supplementary analytical layer and portfolio enhancement  

This approach ensures both **technical rigor** and **professional presentation**.

---

---

## Author & Acknowledgment

**Author:**  
Deepan Mehta  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on preparing prediction outputs for visualization in an interactive dashboard by structuring, aggregating, and formatting data.

The methodology aligns with IBM Skills Network instructional labs on dashboard development and data presentation.

Special acknowledgment is given to:

- Yan Luo  
- Jeff Grossman  

---

## Project Context

This notebook represents the final data preparation stage within the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Model Development  
- Model Evaluation  
- API Integration  
- Prediction Pipeline  
- **Dashboard Preparation (Visualization Layer)**  

---

## Notes

All code and explanations have been independently structured to reflect a production-ready workflow, ensuring seamless integration with the Shiny dashboard.

---